# Piecewise Regression 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import piecewise_regression
import os
import re
import matplotlib.patheffects as pe

seed=1234
np.random.seed(seed)

%matplotlib inline

In [3]:
period_map = {1: 'data_1Y', 2: 'data_2Y', 3: 'data_1Y', 4: 'data_2Y'}
status_map = {1: 'Normal', 2: 'Normal', 3: 'Pre DM', 4: 'Pre DM'}

---

In [1]:
def nonlinear_func(dat_type, 
                   realvalue='FBG_realvalue', feature='FBG',
                   n_breakpoints=1,
                   savepath='./OUT(XAI)/PR/1Y(Normal)_FBG.jpg',
                   offset_frac=0.02):
    data = pd.read_csv(f'./OUT(XAI)/{period_map[dat_type]}/shap_values/[lightgbm]({status_map[dat_type]}).csv')
    x = np.asarray(data[realvalue], dtype=float)
    y = np.asarray(data[feature], dtype=float)

    # 적합
    pw_fit = piecewise_regression.Fit(x, y, n_breakpoints=n_breakpoints)
    results = pw_fit.get_results()

    # R^2 계산
    rss = float(results["rss"])
    tss = float(np.sum((pw_fit.yy - np.mean(pw_fit.yy))**2))
    r2 = 1 - rss / tss

    # breakpoint 
    bps = [results["estimates"][f"breakpoint{i}"]["estimate"]
           for i in range(1, n_breakpoints+1)]

    # x-axis offset 
    xmin, xmax = min(x), max(x)
    span = xmax - xmin
    offsets = np.linspace(-offset_frac, offset_frac, len(bps)) * span
    label_xs = [bp + dx for bp, dx in zip(bps, offsets)]

    # 플롯
    pw_fit.plot_data(s=5, color='lightgray', alpha=0.4)
    # 2)
    x_line = np.linspace(x.min(), x.max(), 600)
    y_line = pw_fit.predict(x_line)
    
    # 3) 
    for lw, a in [(10, 0.08), (8, 0.10), (6, 0.12), (4, 0.16)]:
        plt.plot(
            x_line, y_line,
            color="#e53935", alpha=a, lw=lw, solid_capstyle="round",
            antialiased=True, zorder=2
        )
    
    # 4) 
    plt.plot(
        x_line, y_line,
        color="#e53935", lw=2.2, alpha=0.95,
        solid_capstyle="round", solid_joinstyle="round",
        antialiased=True, zorder=3,
        label=f"piecewise, $R^2={r2:.2f}$"
    )
    
    # 5) breakpoint line
    for bp in bps:
        y_bp = pw_fit.predict(np.array([bp]))[0]

        ## 1) 
        dash = (0, (2, 2))  # (offset, (on, off))

        ## 2) 
        for lw, a in [(8, 0.06), (6, 0.08), (4, 0.12)]:
            glow = plt.axvline(bp, color="#2e7d32", linestyle=dash, lw=lw, alpha=a, zorder=1)
            glow.set_path_effects([pe.Stroke(linewidth=lw+2, foreground="white", alpha=0.25), pe.Normal()])

        ## 3) 
        line = plt.axvline(bp, color="#2e7d32", linestyle=dash, lw=1.8, alpha=0.95, zorder=2)
        ### 
        try:
            line.set_dash_capstyle('round') 
        except Exception:
            pass

        ## 4) 
        plt.plot(bp, y_bp, "o", ms=4, color="#2e7d32", zorder=3)
    
    # 6) 
    plt.grid(True, linewidth=0.6, alpha=0.15)
    plt.xlabel('FBG (mg/dL)')
    plt.ylabel(f"SHAP value for {feature}")

    ymin, ymax = plt.ylim()
    dy = (ymax - ymin) * 0.015
    
    for i, (bp, lx) in enumerate(zip(bps, label_xs), start=1):
        y_bp = pw_fit.predict(np.array([bp]))[0]
        plt.text(lx, y_bp + dy, f"{bp:.0f}",
                 va='bottom', ha='center',
                 fontsize=10, fontweight='bold', color='black')

    plt.tick_params(axis='y', which='both', length=0)
    plt.tick_params(axis='x', which='both', length=0)
    
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['left'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
#    plt.gca().axes.yaxis.set_visible(False)
    # save
    plt.savefig(savepath, bbox_inches='tight', dpi=300)
    plt.close()

# 1Y_Prediabites progression_FBG¶

In [4]:
nonlinear_func(1, realvalue='FBG_realvalue', feature='FBG',
              n_breakpoints=1,
              savepath='./OUT(XAI)/PR/1Y(Normal)_FBG.jpg',)

1Y_Prediabites progression_HDL

In [5]:
nonlinear_func(1, realvalue='HDL_realvalue', feature='HDL',
              n_breakpoints=2,
              savepath='./OUT(XAI)/PR/1Y(Normal)_HDL.jpg',)

2Y_Prediabites progression_FBG¶

In [6]:
nonlinear_func(2, realvalue='FBG_realvalue', feature='FBG',
              n_breakpoints=1,
              savepath='./OUT(XAI)/PR/2Y(Normal)_FBG.jpg',)

2Y_Prediabites progression_HDL

In [7]:
nonlinear_func(2, realvalue='HDL_realvalue', feature='HDL',
              n_breakpoints=2,
              savepath='./OUT(XAI)/PR/2Y(Normal)_HDL.jpg',)

---

# 2Y_Diabites progression_FBG

In [8]:
nonlinear_func(4, realvalue='FBG_realvalue', feature='FBG',
              n_breakpoints=2,
              savepath='./OUT(XAI)/PR/2Y(PreDM)_FBG.jpg',)

# 2Y_Diabites progression_HDL¶

In [9]:
nonlinear_func(4, realvalue='HDL_realvalue', feature='HDL',
              n_breakpoints=2,
              savepath='./OUT(XAI)/PR/2Y(PreDM)_HDL.jpg',)